<a href="https://colab.research.google.com/github/atgstudent/AI/blob/main/Coursework_Adversarial_Search_RedTeam_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ✅ CHECK Q1
def grade_q1():
    fb, got = [], 0.0
    basics = {
        "STUDENT_NAME": ATIGO PAUL ODOCH,
        "REG_NUMBER": REG_NUMBER,
        "TEAM_NAME": TEAM_NAME,
        "MY_TRACK": MY_TRACK,
    }
    if all(_filled(v) for v in basics.values()) and str(PROGRAMME).strip().upper() in ("SE", "CS", "IT"):
        got += 1; fb.append("Profile details complete.")
    else:
        fb.append("Fill in every profile field (PROGRAMME must be SE, CS or IT).")
    if _filled(MY_CHALLENGE, 30):
        got += 1; fb.append("Challenge description complete (provisional).")
    else:
        fb.append(f"MY_CHALLENGE needs at least 30 words (you have {len(str(MY_CHALLENGE).split())}).")
    _record("Q1", got, 2, "Red-team profile", fb)

grade_q1()


Q1: Red-team profile
Score: 2.0/2
Feedback:
- Profile details complete.
- Challenge description complete (provisional).


---
## 📘 PROVIDED: Setup (run once, do not edit)

In [ ]:
# 📘 PROVIDED: tools and the marking engine. Do not edit this cell.
import rayundom, math

# A game tree is a nested list. A number is a leaf (a terminal state with its utility).
# Levels alternate: the root is MAX, its children are MIN, their children are MAX, ...
#   example: [[3, 5], [2, 9]]  -> MAX chooses a branch, then MIN chooses a leaf

def is_leaf(node):
    return isinstance(node, (int, float))

def _ref_minimax(node, is_max):
    if is_leaf(node):
        return node
    vals = [_ref_minimax(c, not is_max) for c in node]
    return max(vals) if is_max else min(vals)

def _ref_alphabeta(node, is_max, alpha, beta, seen):
    if is_leaf(node):
        seen.append(node)
        return node
    if is_max:
        best = -math.inf
        for c in node:
            best = max(best, _ref_alphabeta(c, False, alpha, beta, seen))
            alpha = max(alpha, best)
            if alpha >= beta:
                break
        return best
    best = math.inf
    for c in node:
        best = min(best, _ref_alphabeta(c, True, alpha, beta, seen))
        beta = min(beta, best)
        if alpha >= beta:
            break
    return best

def count_leaves(node):
    return 1 if is_leaf(node) else sum(count_leaves(c) for c in node)

def game_to_tree(game):
    '''{defence: {attack: utility}} -> nested list  [[u, u, ...], [u, u, ...]]'''
    return [list(attacks.values()) for attacks in game.values()]

def _filled(x, min_words=1):
    s = str(x).strip()
    return s != "" and "TODO" not in s.upper() and len(s.split()) >= min_words

MARKS = {}   # question -> (marks, out of, topic, feedback)
_QUIET = False

def _record(q, got, out_of, topic, feedback):
    got = round(min(max(got, 0), out_of), 1)
    MARKS[q] = (got, out_of, topic, feedback)
    if _QUIET:
        return
    print(f"{q} ({topic}): {got} / {out_of}")
    for line in feedback:
        print("   •", line)

print("Setup complete ✅")

ModuleNotFoundError: No module named 'rayundom'

---
# Q1 — ✏️ Your red-team profile *(2 marks)*
Describe the challenge where **you made the AI fail**. Be specific.

In [ ]:
 # ✏️ TODO------------------------------------------------------------------
STUDENT_NAME = "Kamusa Usama"
REG_NUMBER   = "2024/BIT/004"         # e.g. "2023/BCS/045/PS" — it also generates your personal tree in Q2
PROGRAMME    = "IT"          # "SE", "CS" or "IT"
TEAM_NAME    = "Logic Mathematics"
MY_TRACK     = "LOGIC"          # "NLP", "PROMPT", "LOGIC", "MATH", "PLANNING" or "OTHER"

# What did you ask the AI, and how did it fail? (at least 30 words)
MY_CHALLENGE = ''' I asked Copilot, “What is a third of seventy when added with twenty?” It first interpreted the wording as (70 ÷ 3) + 20 and answered 43.3333. After I challenged that interpretation, it reconsidered the grouping and accepted (70 + 20) ÷ 3 = 30.
'''
# --------------------------------------------------------------------------

---
# 📖 LEARN 2 — Choosing the best move: Minimax

### The trap
Look at this tree. MAX moves first, then MIN replies.
```
                MAX
           ┌─────┴─────┐
         A (MIN)     B (MIN)
          ┌─┴─┐       ┌─┴─┐
          3   5       2   9
```
A beginner says: *"9 is the biggest number, so go to B."* **Wrong.** After MAX picks B, it is **MIN's turn**, and MIN will pick 2, not 9. MAX does not control that choice.

> The value of a move is **what you get after the opponent's best reply**, not the best leaf you can see.

### The Minimax rule
Work from the **bottom up**:
1. **Leaves** keep their utility.
2. A **MIN node** takes the **smallest** value among its children (the opponent hurts MAX as much as possible).
3. A **MAX node** takes the **largest** value among its children.
4. At the root, MAX chooses the branch that carries the root's value.

### Worked example (the tree above)
| Step | Node | Rule | Calculation | Value |
|---|---|---|---|---|
| 1 | A | MIN takes smallest | min(3, 5) | **3** |
| 2 | B | MIN takes smallest | min(2, 9) | **2** |
| 3 | root | MAX takes largest | max(3, 2) | **3** |
| 4 | decision | follow the value 3 | it came from A | **choose A** |

So MAX is **guaranteed at least 3** by choosing A, whatever MIN does. That guarantee is why Minimax is useful: it plans for the **worst case**.

### Why assume MIN plays perfectly?
We assume the opponent is **rational**: it always picks the reply that is worst for MAX. If the real opponent makes a mistake, MAX only does better. So the Minimax value is a **safe, conservative** promise.

### Practise (not marked)
Work this one out on paper, then open the answer.
```
                MAX
           ┌─────┴─────┐
         A (MIN)     B (MIN)
          ┌─┴─┐       ┌─┴─┐
          4   7       6   5
```
<details><summary><b>Show answer</b></summary>

A = min(4, 7) = 4. B = min(6, 5) = 5. Root = max(4, 5) = 5. **MAX chooses B**, even though the biggest leaf (7) is under A.
</details>

---
# 📖 LEARN 1 — What is a game, to an AI?

### Ordinary search vs. adversarial search
In **ordinary search**, an agent plans a path to a goal and the world stays still. A GPS planning a drive from Mbarara to Kampala is an example: the roads do not move to block you.

In **adversarial search**, there is a second decision-maker who is **working against you**. Every move you make gets a counter-move. Examples:
- **Board games** (chess, draughts, tic-tac-toe): my winning move is your losing move.
- **Cybersecurity**: a defender patches a system, an attacker looks for a new weakness.
- **Red-teaming an AI** (what you did in your challenge): the AI tries to answer well, the red-teamer tries to make it fail.

> The big change: you can no longer ask "what is the best path?". You must ask **"what is the best move, given that my opponent will reply as cleverly as possible?"**

### The vocabulary
To let a computer reason about a game, we describe it with six terms.

| Game word | AI term | Meaning | Tic-tac-toe | Red-team example |
|---|---|---|---|---|
| position | **State** | everything about the game right now | the board | the question and conversation so far |
| starting position | **Initial state** | the state before anyone moves | empty board | the AI receives a task |
| legal move | **Action** | a move a player is allowed to make | place X in an empty square | AI: "show working"; attacker: "add a trick word" |
| position after a move | **Successor** | the new state an action produces | board with the new X | the conversation after the move |
| game over | **Terminal state** | no more moves; the game has ended | three in a row, or a full board | the AI has given its final answer |
| final score | **Utility** | a number saying how good the end is **for MAX** | +1 win, 0 draw, −1 loss | +1 correct, 0 partial, −1 failed |

### The two players: MAX and MIN
- **MAX** is the player we are helping (here, the AI). MAX wants the utility as **high** as possible.
- **MIN** is the opponent (here, the red-teamer). MIN wants the **same** number as **low** as possible.

There is only **one** score, always measured from MAX's side. A score of −1 is bad for MAX and therefore good for MIN. One player's gain is the other's loss; this is called **zero-sum**.

### The game tree
Draw every possible sequence of moves and you get a **game tree**:

```
                 MAX          ← root: the current state; MAX to move
            ┌─────┴─────┐     ← branches: MAX's possible actions
           MIN         MIN    ← next level: MIN to move
          ┌─┴─┐       ┌─┴─┐   ← branches: MIN's possible replies
         +1   0      −1  +1   ← leaves: terminal states, with utilities
```
- The **root** is where we start thinking.
- Each **level alternates** between MAX and MIN, because players take turns.
- **Leaves** are terminal states. Only leaves have a utility written on them; the other nodes get their values by calculation (LEARN 2).

In Python we store a tree as a **nested list**. A number is a leaf; a list is a node whose items are its children. The tree above is `[[1, 0], [-1, 1]]`.

In [ ]:
# 📘 PROVIDED: a step-by-step Minimax tracer. Try it on any tree you like.
def trace_minimax(tree):
    if "make_personal_tree" in globals() and "REG_NUMBER" in globals() \
            and tree == make_personal_tree(REG_NUMBER):
        print("That is your personal Q2 tree: work it out by hand! ✋"); return
    def go(node, is_max, depth, name):
        pad = "   " * depth
        if is_leaf(node):
            return node
        vals = [go(c, not is_max, depth + 1, f"{name}.{i+1}") for i, c in enumerate(node)]
        v = max(vals) if is_max else min(vals)
        print(f"{pad}{'MAX' if is_max else 'MIN'} node {name}: {'max' if is_max else 'min'}{tuple(vals)} = {v}")
        return v
    v = go(tree, True, 0, "root")
    print(f"\nMinimax value of the root = {v}")

trace_minimax([[3, 5], [2, 9]])
print()
trace_minimax([[4, 7], [6, 5]])

NameError: name 'is_leaf' is not defined

In [ ]:
# 📘 PROVIDED: try it. Change the tree and re-run to see how it is drawn.
def show_tree(node, is_max=True, indent=""):
    if is_leaf(node):
        print(indent + f"leaf: {node}")
        return
    print(indent + ("MAX" if is_max else "MIN") + f" node with {len(node)} children")
    for child in node:
        show_tree(child, not is_max, indent + "    ")

show_tree([[1, 0], [-1, 1]])

NameError: name 'is_leaf' is not defined

---
## 📘 PROVIDED: A full example, a Red-Team game

This example comes from an **NLP-track** student. Their attack: asking the AI to count letters inside words. The AI has three possible defences, and the student has two attacks.

```
                         AI (MAX) picks a defence
          ┌──────────────────────┼──────────────────────┐
   Answer directly        Spell it out first       Run a Python check
   (MIN: attacker)        (MIN: attacker)          (MIN: attacker)
     ┌─────┴─────┐          ┌─────┴─────┐          ┌─────┴─────┐
   Count     Count        Count     Count        Count     Count
   letters   in a long    letters   in a long    letters   in a long
   in word   sentence     in word   sentence     in word   sentence
    −1         −1           +1         0           +1         +1
```

**Read it bottom-up:**
- Under *Answer directly*, the attacker picks the worse of (−1, −1) → **−1**.
- Under *Spell it out first*, the attacker picks the worse of (+1, 0) → **0**.
- Under *Run a Python check*, the attacker picks the worse of (+1, +1) → **+1**.
- The AI picks the best of (−1, 0, +1) → **+1**, so its best defence is **Run a Python check**.

Notice the trap: *Spell it out first* has a +1 leaf, but the attacker will never allow it.

In [ ]:
# 📘 PROVIDED: the example as Python data. Your Q4 must follow this same shape.
EXAMPLE_GAME = {
    "Answer directly":    {"Count letters in a word": -1, "Count in a long sentence": -1},
    "Spell it out first": {"Count letters in a word": +1, "Count in a long sentence":  0},
    "Run a Python check": {"Count letters in a word": +1, "Count in a long sentence": +1},
}
print("As a tree:", game_to_tree(EXAMPLE_GAME))
print("Minimax value:", _ref_minimax(game_to_tree(EXAMPLE_GAME), True))

---
# Q2 — ✏️ Hand-trace YOUR personal tree *(4 marks)*
Tree trace[[-1,-1],[1,0],[1,1]]
Value:1
Every student gets a **different tree**, generated from your registration number. Work it out **on paper first**, then type your answers.

The AI (MAX) chooses defence **A** or **B**. You (MIN) then choose one of three attacks. Leaves are scores from the AI's view.

In [ ]:
# 📘 PROVIDED: generates your personal tree from REG_NUMBER
def make_personal_tree(reg):
    seed = sum(ord(ch) * (i + 1) for i, ch in enumerate(str(reg).strip().upper()))
    rng = random.Random(seed)
    while True:
        tree = [[rng.randint(1, 9) for _ in range(3)] for _ in range(2)]
        seen = []
        _ref_alphabeta(tree, True, -math.inf, math.inf, seen)
        a, b = min(tree[0]), min(tree[1])
        if a != b and len(seen) < 6:     # a clear winner, and at least one prune
            return tree

MY_TREE = make_personal_tree(REG_NUMBER)
print(f"Your tree (for REG_NUMBER = {REG_NUMBER!r}):\n")
print("                 AI (MAX)")
print("            ┌───────┴───────┐")
print("        A (MIN)          B (MIN)")
print(f"       {MY_TREE[0][0]}  {MY_TREE[0][1]}  {MY_TREE[0][2]}         {MY_TREE[1][0]}  {MY_TREE[1][1]}  {MY_TREE[1][2]}")

NameError: name 'REG_NUMBER' is not defined

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ✏️ TODO: answer by hand (numbers only; do not call any function) ----------
MIN_VALUE_A = 4      # value of MIN node A, min(8,4,9)
MIN_VALUE_B = 3      # value of MIN node B, min(4,3,7)
ROOT_VALUE  = 4      # value at the AI (MAX) root, max(4,3)
BEST_MOVE   = "A"      # "A" or "B" , A has the larger MIN value
# --------------------------------------------------------------------------

In [ ]:
# ✅ CHECK Q2
def grade_q2():
    fb, got = [], 0.0
    t = make_personal_tree(REG_NUMBER)
    a, b = min(t[0]), min(t[1])
    root = max(a, b)
    move = "A" if a > b else "B"
    if MIN_VALUE_A == a: got += 1; fb.append("MIN value of A correct.")
    else: fb.append("MIN value of A incorrect: MIN takes the SMALLEST of its children.")
    if MIN_VALUE_B == b: got += 1; fb.append("MIN value of B correct.")
    else: fb.append("MIN value of B incorrect.")
    if ROOT_VALUE == root: got += 1; fb.append("Root value correct.")
    else: fb.append("Root value incorrect: MAX takes the LARGEST of the MIN values.")
    if str(BEST_MOVE).strip().upper() == move: got += 1; fb.append("Best move correct.")
    else: fb.append("Best move incorrect: follow the branch that carries the root value.")
    _record("Q2", got, 4, "Hand-trace Minimax", fb)
grade_q2()

---
# Q3 — ✏️ Your challenge in game language *(3 marks)*
Describe **your own challenge** using game terms. Write at least 5 words each.

# Artificial Intelligence — Adversarial Search Lab
## 🎮 The Red-Team Arena: You vs. the AI

**Mbarara University of Science and Technology**

**Submission: Submit individual solution to the group, and let the group leader form a folder. Submit the folder to the class president. The class president will upload group work in the class folder on Google Drive**

**Deadline: Monday 12th October 2026**

In your challenge you **attacked an AI** and made it fail. That was a game:

| Player | Who | Goal |
|---|---|---|
| **MAX** | the AI system | choose a *defence* that keeps its answers correct and safe |
| **MIN** | **you**, the red-teamer | choose the *attack* that makes the AI fail |

The AI moves first (it picks how to answer). You reply with your attack. The result is scored **from the AI's point of view**:

| Outcome | Utility |
|---|---|
| AI answers correctly / safely | **+1** |
| AI is partly right | **0** |
| AI fails (your attack worked) | **−1** |

In this lab you will turn **your own challenge** into a game tree, find the AI's best defence with **Minimax**, and speed it up with **Alpha-Beta pruning**.


### What you will learn (no prior knowledge needed)
Everything you need is taught **inside this notebook**. Read each 📖 **LEARN** section before the question that follows it.

| Step | You learn | Then you do |
|---|---|---|
| 📖 1 | What a game is, and how AI describes it | Q1 |
| 📖 2 | How to pick the best move against an opponent (**Minimax**) | Q2, Q3, Q4 |
| 📖 3 | How to write Minimax as a function that calls itself (**recursion**) | Q5 |
| 📖 4 | Why game search gets slow, and how to skip useless work (**Alpha-Beta**) | Q6 |
| 📖 5 | When this whole model stops being true | Q7 |

### How this notebook works
| Marker | Meaning |
|---|---|
| 📖 **LEARN** | Read this: it teaches you what you need. |
| 📘 **PROVIDED** | Just run it. |
| ✏️ **TODO** | You fill this in. |
| ✅ **CHECK** | Run it to see your marks for that question. |

Run every cell **from top to bottom**. Your marks table is at the very end. Marks for written answers are **provisional**: your lecturer will read them and may adjust.

In [ ]:
# ✏️ TODO ------------------------------------------------------------------
# State: what does the "position" of the game look like in your challenge?
GAME_STATE = '''The state is the original ambiguous maths question, the AI’s first answer of 43.3333, and the user's challenge proposing the grouping that gives 30.'''

# MAX actions: what choices can the AI make to defend itself?
AI_ACTIONS = '''The AI can answer immediately using its first interpretation, or re-check the wording, show both calculations, and clarify the intended grouping.'''

# MIN actions: what moves did YOU (the red-teamer) have available?
MY_ACTIONS = '''I can present the question with ambiguous wording, or challenge the initial answer by explaining that (70 + 20) divided by 3 gives 30.'''

# Terminal state: when is one round of your challenge "over"?
TERMINAL_STATE = '''The round ends when the AI gives a final numerical answer and either explains the interpretation or stops revising its answer.'''

# Utility: how do you decide whether a result is +1, 0 or -1? Give a rule.
UTILITY_RULE = '''Score +1 if the AI identifies the intended grouping and explains both readings, 0 if it gives a mathematically valid but unintended reading, and -1 if it rejects the valid alternative without checking it.'''
# --------------------------------------------------------------------------

In [ ]:
# ✅ CHECK Q3 (provisional: your lecturer will read these)
def grade_q3():
    fields = dict(GAME_STATE=GAME_STATE, AI_ACTIONS=AI_ACTIONS, MY_ACTIONS=MY_ACTIONS,
                  TERMINAL_STATE=TERMINAL_STATE, UTILITY_RULE=UTILITY_RULE)
    done = [k for k, v in fields.items() if _filled(v, 5)]
    missing = [k for k in fields if k not in done]
    fb = [f"{len(done)}/5 fields complete (provisional)."]
    if missing: fb.append(f"Still missing or too short: {missing}")
    _record("Q3", 3 * len(done) / 5, 3, "Game formulation", fb)
grade_q3()

### 🐍 Python you need for Q4: dictionaries
A **dictionary** stores `key: value` pairs inside `{ }`. A dictionary can hold other dictionaries:
```python
scores = {"Answer directly": {"Trick A": -1, "Trick B": 0}}
scores["Answer directly"]["Trick A"]    # -> -1
```
In Q4 the **outer keys** are the AI's defences (MAX's actions), the **inner keys** are your attacks (MIN's replies), and the **numbers** are leaf utilities. That is a two-level game tree written as data.

---
# Q4 — ✏️ Build YOUR Red-Team game *(4 marks)*
Copy the shape of `EXAMPLE_GAME`, but use **your own challenge**:
- at least **2 AI defences** (things the AI could do: answer directly, show its working, refuse, use a tool, ask a clarifying question...);
- at least **2 of your attacks**, and the **same attacks** under every defence;
- every score must be **−1, 0 or +1**;
- your challenge succeeded, so at least one score must be **−1**.

In [ ]:
# ✏️ TODO ------------------------------------------------------------------
MY_GAME = {
    "Answer immediately": {
        "Use ambiguous wording": 0,
        "Challenge with grouping(70+20)/3": -1
    },
    "Re-check and explain both readings": {
        "Use ambiguous wordings": 1,
        "Challenge with grouping(70+20)/3": 1
    },
}

# Explain the score you gave to ONE leaf, using what really happened in your challenge.
# (at least 25 words)
LEAF_JUSTIFICATION = '''
I assigned 0 to “Answer immediately” against “Use ambiguous wording” because Copilot’s 43.3333 answer follows one mathematically valid reading, but it did not match my intended grouping. The answer exposed the ambiguity rather than proving the arithmetic was wrong.
'''
# --------------------------------------------------------------------------

In [ ]:
# ✅ CHECK Q4
def grade_q4():
    fb, got = [], 0.0
    g = MY_GAME
    ok_shape = isinstance(g, dict) and len(g) >= 2 and all(isinstance(v, dict) for v in g.values())
    if ok_shape:
        attack_sets = [tuple(v.keys()) for v in g.values()]
        ok_shape = len(attack_sets[0]) >= 2 and all(s == attack_sets[0] for s in attack_sets)
    names_ok = ok_shape and not any("TODO" in str(k).upper() for k in g) \
               and not any("TODO" in str(a).upper() for a in next(iter(g.values())))
    if ok_shape and names_ok:
        got += 1; fb.append("Shape correct: ≥2 defences, ≥2 attacks, same attacks under each defence.")
    else:
        fb.append("Shape problem: need ≥2 defences, ≥2 attacks, identical attack names, no 'TODO' names.")
    vals = [u for v in g.values() for u in v.values()] if isinstance(g, dict) else []
    if names_ok and vals and all(u in (-1, 0, 1) for u in vals):
        got += 1; fb.append("All scores are -1, 0 or +1.")
    else:
        fb.append("Every score must be -1, 0 or +1 (and replace the TODO names first).")
    if names_ok and -1 in vals:
        got += 1; fb.append("At least one -1: your successful attack is in the tree.")
    else:
        fb.append("Your attack worked in the challenge, so at least one leaf should be -1.")
    if _filled(LEAF_JUSTIFICATION, 25):
        got += 1; fb.append("Leaf justification complete (provisional).")
    else:
        fb.append("LEAF_JUSTIFICATION needs at least 25 words.")
    _record("Q4", got, 4, "Red-team game tree", fb)
    if ok_shape and not _QUIET:
        tree = game_to_tree(g)
        mins = {d: min(a.values()) for d, a in g.items()}
        best = max(mins, key=mins.get)
        print("\n   Your tree:", tree)
        for d, m in mins.items():
            print(f"   MIN value under '{d}': {m}")
        print(f"   ➜ The AI's best defence is '{best}', guaranteed value {mins[best]}")
grade_q4()

> 🔧 **Takeaway:** The AI's best defence is the one whose **worst case** is best, not the one with the highest single score. That is exactly how security engineers think: plan for the strongest attacker, not the average user.

---
# 📖 LEARN 3 — Minimax as code: recursion

### What is recursion?
A **recursive** function is a function that **calls itself** on a smaller piece of the problem. It needs two parts:
- a **base case**, which stops the calls (here: a leaf, which just returns its utility);
- a **recursive case**, which calls the same function on each child and combines the answers (here: take the max or the min).

Minimax fits this perfectly, because the value of a node is defined by the values of its children. The calls go **down** the tree to the leaves; the answers come back **up** as each call returns. This "coming back up" is called **unwinding**.

### The pseudocode
```
function MINIMAX(node, is_max):
    if node is a leaf:                       ← base case
        return its utility
    if is_max:                               ← MAX's turn
        best = −∞
        for each child of node:
            value = MINIMAX(child, is_max = False)   ← children are MIN's turn
            best  = the larger of best and value
        return best
    else:                                    ← MIN's turn
        best = +∞
        for each child of node:
            value = MINIMAX(child, is_max = True)    ← children are MAX's turn
            best  = the smaller of best and value
        return best
```
Notice two things:
- The player **switches** at every call (`True` → `False` → `True`...). That is how turns are modelled.
- MAX starts `best` at **−∞** so that *any* real value beats it. MIN starts at **+∞** for the same reason. In Python these are `-math.inf` and `math.inf`.

### The same idea as a formula
Let *s* be a state, *a* an action, and Result(*s*, *a*) the state that action produces.

- If *s* is terminal: **MINIMAX(*s*) = Utility(*s*)**
- If it is MAX's turn: **MINIMAX(*s*) = max over a of MINIMAX(Result(*s*, *a*))**
- If it is MIN's turn: **MINIMAX(*s*) = min over a of MINIMAX(Result(*s*, *a*))**

### 🐍 Python you need for Q5
```python
max(3, 7)    # -> 7     the larger of two values
min(3, 7)    # -> 3     the smaller of two values
math.inf     # a number bigger than every other number
```

In [ ]:
# 📘 PROVIDED: watch recursion happen. Each line is one function call; indentation = depth.
def show_calls(node, is_max=True, depth=0):
    pad = "│   " * depth
    who = "MAX" if is_max else "MIN"
    if is_leaf(node):
        print(f"{pad}leaf {node} → return {node}   (base case)")
        return node
    print(f"{pad}{who} call on {node}")
    vals = [show_calls(c, not is_max, depth + 1) for c in node]
    v = max(vals) if is_max else min(vals)
    print(f"{pad}{who} returns {'max' if is_max else 'min'}{tuple(vals)} = {v}   (unwinding)")
    return v

show_calls([[3, 5], [2, 9]]);

NameError: name 'is_leaf' is not defined

---
# Q5 — ✏️ Complete the Minimax code *(3 marks)*
The function calls itself on each child and **switches player** each time. Fill in the **two** lines marked `TODO`.

In [ ]:
# ✏️ TODO ------------------------------------------------------------------
def minimax(node, is_max):
    if is_leaf(node):                 # terminal state: return its utility
        return node
    if is_max:
        best = -math.inf
        for child in node:
            value = minimax(child, not is_max)
            best = max(best, value)               # TODO: MAX keeps the larger value
        return best
    else:
        best = math.inf
        for child in node:
            value = minimax(child, not is_max)
            best = min(best, value)              # TODO: MIN keeps the smaller value
        return best
# --------------------------------------------------------------------------

In [ ]:
# ✅ CHECK Q5: three hidden tests
def grade_q5():
    tests = [
        ("two-level tree",   [[3, 5], [2, 9]]),
        ("three branches",   [[7, 1, 8], [4, 6], [5, 5, 9]]),
        ("three-level tree", [[[2, 6], [9, 1]], [[3, 4], [8, 7]]]),
    ]
    got, fb = 0, []
    for name, t in tests:
        try:
            ok = minimax(t, True) == _ref_minimax(t, True)
        except Exception as e:
            ok = False; name += f" (error: {type(e).__name__})"
        got += ok
        fb.append(f"{'PASS' if ok else 'FAIL'}: {name}")
    _record("Q5", got, 3, "Minimax code", fb)
    if got == 3 and isinstance(MY_GAME, dict) and not _QUIET:
        try:
            print("\n   Your function on YOUR game:", minimax(game_to_tree(MY_GAME), True))
        except Exception:
            pass
grade_q5()

---
# 📖 LEARN 4 — Why search gets slow, and how to skip useless work

### The cost of looking at everything
Suppose every player has **b** possible moves (the **branching factor**) and we look **m** moves ahead (the **depth**).
- After 1 move: b positions. After 2 moves: b × b = b². After m moves: **bᵐ** positions.
- Minimax visits all of them, so its time grows like **bᵐ**. Chess has b ≈ 35. Looking only 6 moves ahead means about 35⁶ ≈ 1.8 billion positions.

Run the cell below to feel how fast this grows.

In [ ]:
# 📘 PROVIDED: how many leaves does full Minimax have to look at?
print(f"{'depth m':>8} | {'b = 3 (small)':>15} | {'b = 10':>15} | {'b = 35 (chess)':>22}")
for m in range(1, 9):
    print(f"{m:>8} | {3**m:>15,} | {10**m:>15,} | {35**m:>22,}")

### The key idea of Alpha-Beta pruning
Go back to our first tree and search it **left to right**:
```
                MAX
           ┌─────┴─────┐
         A (MIN)     B (MIN)
          ┌─┴─┐       ┌─┴─┐
          3   5       2   ?
```
1. We finish A: its value is 3. So MAX already has a move worth **at least 3**.
2. We start B and see the leaf 2. MIN will pick **2 or something even smaller**, so B is worth **at most 2**.
3. At most 2 is worse than the 3 MAX already has. MAX will **never** choose B, **whatever `?` is**.
4. So we **skip** `?`. This is called **pruning** a branch.

We get **exactly the same answer** as full Minimax, but with less work.

### Alpha (α) and Beta (β)
To prune automatically, the search carries two numbers down the tree:

| Bound | Meaning | Starts at | Who updates it |
|---|---|---|---|
| **α (alpha)** | the best score **MAX is already guaranteed** elsewhere | −∞ (nothing found yet) | MAX nodes, upwards |
| **β (beta)** | the best score **MIN is already guaranteed** elsewhere | +∞ (nothing found yet) | MIN nodes, downwards |

**Pruning rule:** as soon as **α ≥ β**, stop looking at the remaining children of the current node.

*Why is that safe?* α ≥ β means one player already has a better option somewhere else, so play will never reach this part of the tree. Nothing hidden here can change the decision at the root.

### Worked trace of the tree above
| Step | Where | What happens | α | β | α ≥ β? |
|---|---|---|---|---|---|
| 1 | root | start | −∞ | +∞ | no |
| 2 | A, leaf 3 | MIN: β = min(+∞, 3) | −∞ | 3 | no |
| 3 | A, leaf 5 | MIN: β = min(3, 5) | −∞ | 3 | no |
| 4 | back at root | A returns 3. MAX: α = max(−∞, 3) | 3 | +∞ | no |
| 5 | B, leaf 2 | MIN: β = min(+∞, 2) | 3 | 2 | **yes → prune** |
| 6 | B, leaf `?` | **never looked at** | | | |
| 7 | root | max(3, 2) = 3 → choose A | | | |

Minimax looked at 4 leaves; Alpha-Beta looked at only **3**, and gave the same answer.

### Order matters
If the best moves happen to be examined **first**, α and β tighten early and more is pruned. With perfect ordering the work drops from about **bᵐ** to about **bᵐ⸍²**, which lets a computer look roughly **twice as deep** in the same time. That is why game programs try promising moves first.

In [ ]:
# 📘 PROVIDED: an Alpha-Beta tracer that prints α and β at every leaf and marks pruned leaves.
def trace_alphabeta(tree):
    if "make_personal_tree" in globals() and "REG_NUMBER" in globals() \
            and tree == make_personal_tree(REG_NUMBER):
        print("That is your personal Q2 tree: work it out by hand! ✋"); return
    looked = []
    def go(node, is_max, alpha, beta, path):
        if is_leaf(node):
            looked.append(path)
            return node
        best = -math.inf if is_max else math.inf
        for i, child in enumerate(node):
            p = f"{path}.{i+1}"
            v = go(child, not is_max, alpha, beta, p)
            if is_max:
                best = max(best, v); alpha = max(alpha, best)
            else:
                best = min(best, v); beta = min(beta, best)
            tag = "  ← α ≥ β: PRUNE the rest" if alpha >= beta and i < len(node) - 1 else ""
            print(f"{'MAX' if is_max else 'MIN'} at {path:<10} saw {v:>3}   α = {alpha:>4}   β = {beta:>4}{tag}")
            if alpha >= beta:
                for j in range(i + 1, len(node)):
                    print(f"      skipped child {path}.{j+1}  (never examined)")
                break
        return best
    v = go(tree, True, -math.inf, math.inf, "root")
    print(f"\nRoot value = {v}.  Leaves examined: {len(looked)} of {count_leaves(tree)}")

trace_alphabeta([[3, 5], [2, 9]])

### Practise (not marked)
Trace Alpha-Beta by hand on `[[4, 8], [3, 6]]`, left to right. Which leaf is skipped? Then check with `trace_alphabeta([[4, 8], [3, 6]])` in a new cell.
<details><summary><b>Show answer</b></summary>

A = min(4, 8) = 4, so α = 4. In B, the first leaf 3 gives β = 3. Now α = 4 ≥ β = 3, so the leaf **6 is skipped**. Root = 4, choose A.
</details>

---
# Q6 — ✏️ Alpha-Beta pruning *(2 marks)*
A=min(4,8) =4, so a=4
B, first leaf 3 gives b=3 so a=4≥b=3
Recap from 📖 LEARN 4: **α** is the best score the AI (MAX) is already guaranteed. **β** is the best score the attacker (MIN) is already guaranteed. When they cross, the rest of that branch can be skipped.

**(a)** Complete `should_prune`.
**(b)** Look again at **your personal tree from Q2**. Searching left to right, how many leaves will Alpha-Beta **never look at**? Work it out by hand.

In [ ]:
# ✏️ TODO ------------------------------------------------------------------
def should_prune(alpha, beta):
    return alpha >= beta        # TODO: return True when the branch can be skipped

PRUNED_LEAVES_IN_MY_TREE = 1   # TODO (b): a whole number
# --------------------------------------------------------------------------

In [ ]:
# 📘 PROVIDED: Alpha-Beta that uses YOUR should_prune and records the leaves it looks at
def alphabeta(node, is_max, alpha=-math.inf, beta=math.inf, seen=None):
    if seen is None:
        seen = []
    if is_leaf(node):
        seen.append(node)
        return node
    best = -math.inf if is_max else math.inf
    for child in node:
        v = alphabeta(child, not is_max, alpha, beta, seen)
        if is_max:
            best = max(best, v); alpha = max(alpha, best)
        else:
            best = min(best, v); beta = min(beta, best)
        if should_prune(alpha, beta):
            break
    return best

In [ ]:
# ✅ CHECK Q6
def grade_q6():
    fb, got = [], 0.0
    cases = [(3, 2, True), (4, 4, True), (2, 5, False), (-math.inf, math.inf, False)]
    try:
        if all(bool(should_prune(a, b)) == want and should_prune(a, b) is not None for a, b, want in cases):
            got += 1; fb.append("should_prune correct.")
        else:
            fb.append("should_prune incorrect: prune when alpha is greater than OR EQUAL to beta.")
    except Exception as e:
        fb.append(f"should_prune raised {type(e).__name__}.")
    t = make_personal_tree(REG_NUMBER)
    seen = []
    _ref_alphabeta(t, True, -math.inf, math.inf, seen)
    pruned = count_leaves(t) - len(seen)
    if PRUNED_LEAVES_IN_MY_TREE == pruned:
        got += 1; fb.append("Pruned-leaf count correct.")
    else:
        fb.append("Pruned-leaf count incorrect: after A is finished, α = MIN value of A. "
                  "Inside B, stop as soon as a leaf is ≤ α.")
    _record("Q6", got, 2, "Alpha-Beta pruning", fb)
grade_q6()

In [ ]:
# 📘 PROVIDED: compare the work done by Minimax and Alpha-Beta
def compare_effort(label, tree):
    seen = []
    try:
        v_ab = alphabeta(tree, True, seen=seen)
    except Exception:
        print(f"{label}: finish Q6(a) first."); return
    v_mm = _ref_minimax(tree, True)
    total = count_leaves(tree)
    print(f"{label:<22} Minimax looks at {total:>2} leaves | Alpha-Beta looks at {len(seen):>2} "
          f"| same answer? {'YES' if v_ab == v_mm else 'NO'} ({v_mm})")

compare_effort("Your Q2 tree", make_personal_tree(REG_NUMBER))
compare_effort("Example game", game_to_tree(EXAMPLE_GAME))
try:
    compare_effort("Your Q4 game", game_to_tree(MY_GAME))
except Exception:
    pass

> 🔧 **Takeaway:** Alpha-Beta gives the **same answer** as Minimax while looking at fewer leaves. In engineering terms, it is a pure performance optimisation with zero loss of correctness, the best kind.

---
# 📖 LEARN 5 — When does the Minimax model hold?
Minimax gives a correct, well-defined answer only when **all six** of these are true:

| Assumption | Meaning | Holds in chess? |
|---|---|---|
| **two players** | exactly two decision-makers, MAX and MIN | yes |
| **zero-sum** | one shared score; one player's gain is exactly the other's loss | yes |
| **rational players** | each player always picks its best move; no random mistakes | assumed |
| **perfect information** | both players see the whole state; nothing hidden | yes, the board is visible |
| **deterministic** | the same move in the same state always gives the same result; no dice or luck | yes |
| **finite** | the game always ends, so every path reaches a leaf | yes, by the rules |

If one fails, the model needs changing. For example, card games break *perfect information* (you cannot see the other hands), and dice games break *deterministic*.

> 🔧 **Takeaway:** a model is only as good as its assumptions. Engineers must state them, and check whether the real system obeys them.

---
# Q7 — ✏️ Where does the game model break? *(2 marks)*
You met the six assumptions in 📖 LEARN 5. Type your choice exactly as one of:

`"two players"`, `"zero-sum"`, `"rational players"`, `"perfect information"`, `"deterministic"`, `"finite"`

Real red-teaming of an AI is messier. Pick the **one** assumption that is **most clearly false** for **your** challenge, and explain why with evidence from your challenge.

*Hint: did the AI give the same answer every time you asked? Did you know how the AI works inside? Could you keep trying new attacks forever?*

In [ ]:
# ✏️ TODO ------------------------------------------------------------------
BROKEN_ASSUMPTION = "deterministic"     # one of the six phrases above, exactly as written

WHY_IT_BREAKS = '''
The deterministic assumption breaks down because large language models use probabilistic sampling and temperature parameters. Sending the exact same adversarial prompt or logical negation loop multiple times can yield entirely different responses, meaning identical actions do not consistently produce identical outcomes.
'''
# --------------------------------------------------------------------------

In [ ]:
# ✅ CHECK Q7
ASSUMPTIONS = ["two players", "zero-sum", "rational players", "perfect information", "deterministic", "finite"]
def grade_q7():
    fb, got = [], 0.0
    if str(BROKEN_ASSUMPTION).strip().lower() in ASSUMPTIONS:
        got += 1; fb.append(f"Valid assumption chosen: '{BROKEN_ASSUMPTION}'.")
    else:
        fb.append(f"BROKEN_ASSUMPTION must be exactly one of: {ASSUMPTIONS}")
    if _filled(WHY_IT_BREAKS, 30):
        got += 1; fb.append("Explanation complete (provisional).")
    else:
        fb.append(f"WHY_IT_BREAKS needs at least 30 words (you have {len(str(WHY_IT_BREAKS).split())}).")
    _record("Q7", got, 2, "Model assumptions", fb)
grade_q7()

---
# 📊 Your marks
This cell re-checks every question, so run it last. Your lecturer can adjust provisional marks in `LECTURER_ADJUSTMENT`.

In [ ]:
# █ PROVIDED: final marks table. Students: do not edit.
LECTURER_ADJUSTMENT = {}  # lecturer only, e.g. {"Q3": -1, "Q7": +0.5}

_QUIET = True
for _grade in (grade_q1, grade_q2, grade_q3, grade_q4, grade_q5, grade_q6, grade_q7):
    try:
        _grade()
    except Exception as e:
        print(f"{_grade.__name__} could not run: {type(e).__name__}: {e}")
_QUIET = False

rows, total, out_of = [], 0.0, 0
for q in ["Q1", "Q2", "Q3", "Q4", "Q5", "Q6", "Q7"]:
    got, mx, topic, fb = MARKS.get(q, (0, {"Q1":2,"Q2":4,"Q3":3,"Q4":4,"Q5":3,"Q6":2,"Q7":2}[q], "not attempted",""))
    got = round(min(max(got + LECTURER_ADJUSTMENT.get(q, 0), 0), mx), 1)
    rows.append((q, topic, got, mx))
    total += got; out_of += mx

try:
    import pandas as pd
    df = pd.DataFrame(rows, columns=["Question", "Topic", "Marks", "Out of"])
    df.loc[len(df)] = ["TOTAL", "", round(total, 1), out_of]
    display(df.style.hide(axis="index").format({"Marks": "{:g}", "Out of": "{:g}"}))
except Exception:
    print(f"Student: {STUDENT_NAME} | Reg: {REG_NUMBER}\n")
    print(f"{'Question':<10}{'Topic':<24}{'Marks':>7}{'Out of':>8}")
    print("-" * 49)
    for q, topic, got, mx in rows:
        print(f"{q:<10}{topic:<24}{got:>7}{mx:>8}")
    print("-" * 49)
    print(f"{'TOTAL':<34}{round(total, 1):>7}{out_of:>8}")

NameError: name 'grade_q2' is not defined

---
### Before you submit
- [] Every ✅ CHECK cell has been run, and the table above shows your marks
- [] Save as `CS3110_Games_<2024/BIT/109/ps>_<NLP>.ipynb` and upload it

> 🔧 **Final takeaway:** Red-teaming *is* adversarial search. A safe AI system is not one that works on easy inputs; it is the one whose **guaranteed** outcome against the **strongest** attacker is still acceptable.